In [ ]:
import wandb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
%pip install skorch -q

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

train['combine'] = (train['prompt'] + " " +
                train['A'] + " " +
                train['B'] + " " +
                train['C'] + " " +
                train['D'] + " " +
                train['E'])

test['combine'] = (test['prompt'] + " " +
                test['A'] + " " +
                test['B'] + " " +
                test['C'] + " " +
                test['D'] + " " +
                test['E']
)
X = train['combine'].values
y = train['answer'].values

In [ ]:
# Encode labels A→0, B→1 etc
le = LabelEncoder()
y_encoded = le.fit_transform(y).astype(np.int64)

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)


In [ ]:

vectorizer = TfidfVectorizer(max_features=10000)
X_train_nn = vectorizer.fit_transform(X_train).toarray().astype(np.float32)
X_val_nn = vectorizer.transform(X_val).toarray().astype(np.float32)

X_train_nn.shape

In [ ]:
import torch
import torch.nn as nn
from skorch import NeuralNetClassifier


# Train simple neural network for my dataset
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2939, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 5)
        )
    def forward(self, x):
        return self.network(x)

In [ ]:
net = NeuralNetClassifier(
    SimpleNN,
    max_epochs=20,
    lr=0.001,
    batch_size=32,
    optimizer=torch.optim.Adam,
    criterion=nn.CrossEntropyLoss,
    iterator_train__shuffle=True,
    verbose=1
)

net.fit(X_train_nn, y_train)
pred3 = net.predict(X_val_nn)

f1 = f1_score(y_val, pred3, average='macro')
acc = accuracy_score(y_val, pred3)
print(f1)
print(acc)
